# MalariaScope — Full-Field Dataset Audit

**Pilot A: NIH/NLM (Bangladesh) + Makerere (Uganda)**

**Purpose:** Inspect public dataset access, labels, source metadata, and leakage risks **before** model testing or training.

**Research only — not a clinical diagnostic tool.** No images are downloaded or models modified by this notebook. The original MalariaScope MobileNetV3 checkpoint stays frozen.

**How to use:** Run the numbered sections in order. The code catches network errors and records what remains unverified. Do not claim an external-test result from this audit.

## 1 — Prepare the notebook

Create a new audit folder. Keep it separate from the previously completed Kaggle training run.

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
from html.parser import HTMLParser
from urllib.parse import urljoin, quote
from urllib.request import Request, urlopen

OUT = Path("/kaggle/working/malariascope_fullfield_audit")
OUT.mkdir(parents=True, exist_ok=True)
print("Audit folder:", OUT)
print("UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

## 2 — Set source links

NIH/NLM has **polygon** and **point** full-field thin-smear subsets. Makerere's dataset is identified by DOI **10.7910/DVN/VEADSE**; only its thin-smear portion is relevant to our current cell classifier.

References: [NIH README](https://data.lhncbc.nlm.nih.gov/public/Malaria/NIH-NLM-ThinBloodSmearsPf/ReadMe.pdf) · [NIH index](https://data.lhncbc.nlm.nih.gov/public/Malaria/NIH-NLM-ThinBloodSmearsPf/index.html) · [Makerere data paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC11719325/) · [Makerere dataset](https://doi.org/10.7910/DVN/VEADSE).

In [ ]:
NIH_ROOT = "https://data.lhncbc.nlm.nih.gov/public/Malaria/NIH-NLM-ThinBloodSmearsPf/"
NIH_POLY = urljoin(NIH_ROOT, "Polygon%20Set/index.html")
NIH_POINT = urljoin(NIH_ROOT, "Point%20Set/index.html")
NIH_LICENSE = urljoin(NIH_ROOT, "Data%20License%20Agreement.docx")
MAKERERE_DOI = "doi:10.7910/DVN/VEADSE"
MAKERERE_API = ("https://dataverse.harvard.edu/api/datasets/:persistentId/"
                "?persistentId=" + quote(MAKERERE_DOI, safe=""))
print("NIH metadata:", NIH_ROOT)
print("Makerere DOI:", MAKERERE_DOI)

## 3 — Check access without downloading datasets

Read only small text index or metadata responses. This is **not** a bulk-image download. Kaggle Internet must be enabled.

In [ ]:
def small_fetch(url, limit=350_000):
    req = Request(url, headers={"User-Agent": "MalariaScopeResearchAudit/1.0"})
    with urlopen(req, timeout=30) as response:
        data = response.read(limit + 1)
    if len(data) > limit:
        raise ValueError("Metadata response exceeded safety size limit")
    return data

class AnchorLinks(HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []
    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            self.hrefs.extend(v for k, v in attrs if k == "href" and v)

def index_links(url):
    parser = AnchorLinks()
    parser.feed(small_fetch(url).decode("utf-8", errors="replace"))
    return parser.hrefs

access = {}
for name, url in [("NIH index", NIH_ROOT), ("NIH polygon", NIH_POLY),
                  ("NIH point", NIH_POINT), ("Makerere API", MAKERERE_API)]:
    try:
        access[name] = {"reachable": True, "bytes": len(small_fetch(url))}
    except Exception as exc:
        access[name] = {"reachable": False, "error": str(exc)[:180]}
    print(f"{name}: {'reachable' if access[name]['reachable'] else 'not accessible'}")

## 4 — Examine the NIH annotation structure

The official NIH README states: **polygon set: 165 fields from 33 patients; point set: 800 fields from 160 patients**. This cell checks live folder counts and, if possible, samples **one annotation text file** (not an image). It only reports aggregate cell categories.

The same Bangladeshi hospital contributed data to the older cropped-cell project. **Patient overlap has not been ruled out**. NIH data should not yet be called an external test.

In [ ]:
nih_subsets = {}
for name, folder in [("polygon", NIH_POLY), ("point", NIH_POINT)]:
    try:
        hrefs = index_links(folder)
        patient_folders = [h for h in hrefs
                           if (h.endswith("/") or h.endswith("/index.html")) and not h.startswith("../")]
        nih_subsets[name] = {"accessible": True,
                              "patient_folders_observed": len(patient_folders),
                              "folder_urls": [urljoin(folder, h) for h in patient_folders]}
        print(name, "patient folders:", len(patient_folders))
    except Exception as exc:
        nih_subsets[name] = {"accessible": False, "error": str(exc)[:180]}
        print(name, "folders not inspected:", str(exc)[:100])

In [ ]:
annotation_probe = {}
for subset, detail in nih_subsets.items():
    if not detail.get("accessible") or not detail.get("folder_urls"):
        annotation_probe[subset] = {"checked": False}
        continue
    try:
        # Access only one public de-identified annotation text file.
        folder = detail["folder_urls"][0]
        gt = urljoin(folder, "GT/index.html")
        txt_files = [h for h in index_links(gt) if h.lower().endswith(".txt")]
        if not txt_files:
            raise ValueError("No .txt annotation entries at sample folder")
        raw = small_fetch(urljoin(gt, txt_files[0]), limit=750_000)
        entries = raw.decode("utf-8", errors="replace").splitlines()
        counts = {}
        for row in entries[1:]:
            fields = row.split(",")
            if len(fields) >= 2:
                label = fields[1].strip()
                counts[label] = counts.get(label, 0) + 1
        annotation_probe[subset] = {"checked": True, "entries_observed": len(entries)-1,
                                     "label_counts_in_one_field": counts}
        print(subset, "sampled cells:", sum(counts.values()), "| classes:", counts)
    except Exception as exc:
        annotation_probe[subset] = {"checked": False, "error": str(exc)[:180]}
        print(subset, "sample unavailable:", str(exc)[:100])

## 5 — Inspect Makerere dataset metadata

Makerere's dataset paper reports **1,000 thin-smear and 3,000 thick-smear fields**. The public datasheet describes JPEG images and YOLO-format annotation files; slide metadata exists for **some**, not necessarily all, images.

Query Harvard Dataverse's *metadata API* only. **Do not download images or assume uninfected RBCs are comprehensively labeled.**

In [ ]:
makerere_audit = {"doi": MAKERERE_DOI, "metadata_accessible": False,
                  "files_listed": 0, "file_examples": [], "issues": []}
try:
    response = json.loads(small_fetch(MAKERERE_API, limit=2_000_000))
    if response.get("status") != "OK":
        raise ValueError("Dataverse returned non-OK status")
    version = (response.get("data") or {}).get("latestVersion") or {}
    files = version.get("files", [])
    rows = []
    for item in files:
        info = item.get("dataFile") or {}
        rows.append({"name": info.get("filename", ""),
                     "content_type": info.get("contentType"),
                     "size_bytes": info.get("filesize"),
                     "restricted": item.get("restricted"),
                     "directory": item.get("directoryLabel", "")})
    makerere_audit.update(metadata_accessible=True, files_listed=len(rows),
                         file_examples=rows[:12],
                         version=str(version.get("versionNumber", "?")))
    print("Dataverse file entries:", len(rows))
    for row in rows[:12]:
        print(" ", row["name"], "|", row["content_type"], "| restricted:", row["restricted"])
except Exception as exc:
    makerere_audit["issues"].append(str(exc)[:220])
    print("Makerere metadata could not be queried automatically.")
    print("Open manually: https://doi.org/10.7910/DVN/VEADSE")
    print("Reason:", str(exc)[:160])

## 6 — Check what still needs confirmation

**NIH:** Read the provider's **Data License Agreement.docx** before bulk download. Verify whether the original cropped-cell patient IDs overlap with this dataset. Polygon and point sets should not share patients without checking.

**Makerere:** Confirm the downloaded label-class mapping, whether **uninfected RBCs** are annotated, whether slide IDs are available, and whether the images/labels are usable. Do not combine thin and thick smears or invent negative labels.

**For both:** Keep patient/slide-separated splits and retain original source/version references.

In [ ]:
checks = {
    "nih_license_read": False,
    "nih_patient_overlap_resolved": False,
    "makerere_label_class_ids_verified": False,
    "makerere_uninfected_rbc_coverage_verified": False,
    "makerere_patient_or_slide_group_ids_verified": False,
    "both_dataset_terms_reviewed": False,
    "model_trained_or_changed": False,
}
for name, passed in checks.items():
    print(("[x]" if passed else "[ ]"), name.replace("_", " "))

## 7 — Save the audit record

Save metadata and unresolved questions locally for the next session. **No model weights, patient IDs, raw images, or clinical results are uploaded to GitHub.**

In [ ]:
# Keep only aggregate NIH records; do not persist patient folder URLs.
nih_safe = {name: {k: v for k, v in info.items() if k != "folder_urls"}
            for name, info in nih_subsets.items()}
audit = {
    "project": "MalariaScope",
    "phase": "uncropped_metadata_audit",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "nih": {"source": NIH_ROOT, "access": access,
            "subsets": nih_safe, "annotation_probes": annotation_probe},
    "makerere": makerere_audit,
    "checks": checks,
    "training_performed": False,
    "bulk_images_downloaded": False
}
target = OUT / "dataset_access_audit.json"
target.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("Saved:", target)
print("Next: verify licenses and labels, then inspect a *small authorized* image sample.")

## 8 — Decision after the audit

**No training yet.** We will first check licensing and actual labels. Then, if the annotation coverage is adequate, evaluate the *frozen* model on reference-cropped cells, clearly separating that test from any later automatic full-field detection experiment.

**We will not claim improved reliability unless new held-out evidence supports it.**